# Fig 2b: Server power over time

Supports Section 2.1, paragraph 2: measured server power has risen over time, and accelerator-rich systems sit in much higher power ranges than general-purpose servers.

**Status:** this notebook currently has the **SPECpower baseline** (general-purpose CPU servers, measured). Two layers are still to come: vendor-rated AI servers (`Data/generated/servers.csv`, not yet built) and MLPerf measured points (`Data/downloaded/mlcommons/power_results.csv`).

**Source:** SPECpower_ssj2008 results export, `Data/downloaded/power_ssj2008-results-20261014-203943.csv` (spec.org/power_ssj2008/results).
- Power = *average watts at 100% of target load* (measured at the wall, whole system), divided by `Nodes` for multi-node results.
- Date = *hardware availability* (month-year), not the test or publication date.
- Server CPUs only (Xeon, EPYC, Opteron, Ampere). The handful of desktop Core/Pentium results are dropped.
- Main series = 2-socket servers, the mainstream data-center configuration. 1-socket and 4- or 8-socket results are kept as context but shown greyed out.

SPECpower measures a Java server workload (ssj), so it's a CPU- and memory-bound load. It isn't the server's rated maximum.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator
from pathlib import Path

DATA = Path('../Data')
OUT = Path('../figures'); OUT.mkdir(exist_ok=True)

# Reference categorical palette: slot 2 (orange) = CPU servers, matching Fig 2a; neutral grey for context
C_CPU, C_CTX = '#eb6834', '#b4b2ab'
TEXT, TEXT_2, GRID = '#0b0b0b', '#52514e', '#e4e3df'

## Load and clean SPECpower

In [ ]:
spec = pd.read_csv(DATA / 'downloaded/power_ssj2008-results-20261014-203943.csv', encoding='latin-1')
spec.columns = [c.strip() for c in spec.columns]  # export has trailing tabs in headers

spec = pd.DataFrame({
    'vendor': spec['Hardware Vendor'].str.strip(),
    'system': spec['System'].str.strip(),
    'hw_avail': pd.to_datetime(spec['HW Avail'], format='%b-%Y', errors='coerce'),
    'nodes': spec['Nodes'],
    'chips': spec['# Chips'],
    'processor': spec['Processor'].str.strip(),
    'form_factor': spec['Form Factor'].str.strip().str.title(),
    'W_100': pd.to_numeric(spec['Average watts @ 100% of target load'], errors='coerce'),
    'W_idle': pd.to_numeric(spec['Average watts @ active idle'], errors='coerce'),
    'psu_rating_W': pd.to_numeric(spec['Power Supply Rating (watts)'], errors='coerce'),
    'url': spec['Disclosure URL'],
})
spec['W_node_100'] = spec['W_100'] / spec['nodes']
spec['W_node_idle'] = spec['W_idle'] / spec['nodes']
spec['sockets_per_node'] = spec['chips'] / spec['nodes']  # '# Chips' is the total across nodes

server_cpu = spec['processor'].str.contains('Xeon|EPYC|Opteron|Ampere', na=False)
print(f'{len(spec)} results; dropping {(~server_cpu).sum()} non-server-CPU results')
spec = spec[server_cpu].dropna(subset=['hw_avail', 'W_node_100'])
spec['config'] = np.select([spec['sockets_per_node'] == 2, spec['sockets_per_node'] == 1], ['2-socket', '1-socket'], '4+ socket')
spec.to_csv(DATA / 'generated/specpower_clean.csv', index=False)
spec['config'].value_counts()

In [ ]:
# Yearly summary for 2-socket servers (the series the text describes)
two = spec[spec['config'] == '2-socket']
yearly = two.groupby(two['hw_avail'].dt.year).agg(
    n=('W_node_100', 'size'), median_W=('W_node_100', 'median'),
    p25_W=('W_node_100', lambda s: s.quantile(0.25)), p75_W=('W_node_100', lambda s: s.quantile(0.75)),
    median_idle_W=('W_node_idle', 'median'))
yearly.round(0)

## Plot

In [ ]:
MIN_N = 5  # only draw the yearly median where a year has at least this many 2-socket results

plt.rcParams.update({'font.family': 'sans-serif', 'font.size': 10, 'axes.edgecolor': GRID,
                     'axes.labelcolor': TEXT_2, 'xtick.color': TEXT_2, 'ytick.color': TEXT_2})
fig, ax = plt.subplots(figsize=(7.5, 4.6), dpi=150)

ctx = spec[spec['config'] != '2-socket']
ax.scatter(ctx['hw_avail'], ctx['W_node_100'], s=14, color=C_CTX, alpha=0.6, edgecolor='none',
           label=f'1-socket and 4+ socket servers (n={len(ctx)})', zorder=2)
ax.scatter(two['hw_avail'], two['W_node_100'], s=16, color=C_CPU, alpha=0.45, edgecolor='none',
           label=f'2-socket servers (n={len(two)})', zorder=3)

y = yearly[yearly['n'] >= MIN_N]
mid = pd.to_datetime(y.index.astype(str) + '-07-01')
ax.plot(mid, y['median_W'], color=C_CPU, lw=2.2, marker='o', ms=5, mfc='white', mew=1.5, zorder=4,
        label=f'2-socket yearly median (years with n ≥ {MIN_N})')

# Direct labels on the first median and the peak median (the latest year is often partial)
for i in (0, int(np.argmax(y['median_W'].values))):
    ax.annotate(f"{y['median_W'].iloc[i]:.0f} W", xy=(mid[i], y['median_W'].iloc[i]),
                xytext=(0, 10), textcoords='offset points', ha='center', fontsize=8, color=TEXT)

ax.set_yscale('log')
ticks = [50, 100, 200, 500, 1000, 2000, 5000]
ax.yaxis.set_major_locator(FixedLocator(ticks)); ax.yaxis.set_minor_locator(NullLocator())
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.set_ylim(40, 6000)
ax.set_xlim(pd.Timestamp('2004-01-01'), pd.Timestamp('2027-01-01'))
ax.xaxis.set_major_locator(mdates.YearLocator(4)); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_ylabel('Measured power per server at 100% load (W, log scale)')
ax.grid(axis='y', color=GRID, lw=0.6); ax.set_axisbelow(True)
for s in ['top', 'right', 'left']:
    ax.spines[s].set_visible(False)
ax.tick_params(length=0)
ax.legend(loc='upper left', frameon=False, fontsize=8.5, labelcolor=TEXT)
ax.set_title('Measured power of general-purpose CPU servers (SPECpower), 2004–2026', loc='left', fontsize=11, color=TEXT)
fig.text(0.01, 0.005, 'Source: SPECpower_ssj2008 published results (SPEC). Date = hardware availability. '
         'Multi-node results divided by node count.', fontsize=6.5, color=TEXT_2)
fig.tight_layout(rect=(0, 0.02, 1, 1))
fig.savefig(OUT / 'fig2b_server_power_specpower.pdf')
fig.savefig(OUT / 'fig2b_server_power_specpower.png', dpi=200)
plt.show()

## Still to add
1. **MLPerf measured points** from `Data/downloaded/mlcommons/power_results.csv` (accelerator servers, measured; few points, often MaxQ/power-capped).
2. **Vendor-rated AI servers** from `Data/generated/servers.csv` (to build; server list seeded from `mlcommons/systems.csv`, power from vendor datasheets).
3. Combined figure on one log axis, with distinct markers for measured vs. rated values.